In [0]:
# Cell 1: settings, city centres, helper for distance
import yaml
from pyspark.sql import functions as F
from pyspark.sql.functions import col

CATALOG = "city_lens"
SILVER = f"{CATALOG}.silver"
TARGET = f"{SILVER}.city_happenings"

# Show all times in Amsterdam local time (roadworks come in UTC)
spark.conf.set("spark.sql.session.timeZone", "Europe/Amsterdam")

with open("../config/cities.yaml") as f:
    cities = yaml.safe_load(f)["cities"]


centres = spark.createDataFrame(
    [(c["key"], float(c["lat"]), float(c["lon"]), float(c["radius_km"])) for c in cities],
    "city_key STRING, centre_lat DOUBLE, centre_lon DOUBLE, radius_km DOUBLE",
)
def distance_km(lat1, lon1, lat2, lon2):
    """Haversine distance between two points, as a Spark expression."""
    dlat = F.radians(lat2 - lat1)
    dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return F.round(2 * 6371 * F.asin(F.sqrt(a)), 2)

In [0]:
# Cell 2: bring both sources into one shape
notices = (
    spark.table(f"{SILVER}.notices")
    .select(
        F.concat(F.lit("sru:"), col("notice_id")).alias("happening_id"),
        F.lit("official_notice").alias("source"),
        "city_key",
        "category",
        "stage",
        "title",
        col("abstract").alias("description"),
        "lat", "lon",
        F.lit("published").alias("time_basis"),
        col("available").alias("published_date"),
        F.lit(None).cast("timestamp_ntz").alias("start_local"),
        F.lit(None).cast("timestamp_ntz").alias("end_local"),
        F.lit("none").alias("closure_level"),
        "url",
        col("modified").cast("timestamp").alias("version_ts"),
    )
)

road = (
    spark.table(f"{SILVER}.road_works")
    .select(
        F.concat(F.lit("ndw:"), col("situation_id")).alias("happening_id"),
        F.lit("ndw_melvin").alias("source"),
        "city_key",
        F.when(col("kind") == "road_event", "road_event").otherwise("road_works").alias("category"),
        F.lit(None).cast("string").alias("stage"),
        "title",
        "description",
        "lat", "lon",
        F.lit("scheduled").alias("time_basis"),
        F.lit(None).cast("date").alias("published_date"),
        col("start_time").cast("timestamp_ntz").alias("start_local"),   # UTC -> Amsterdam wall-clock
        col("end_time").cast("timestamp_ntz").alias("end_local"),
        "closure_level",
        F.lit("https://melvin.ndw.nu/public").alias("url"),
        col("version_time").alias("version_ts"),
    )
)

happenings = (
    notices.unionByName(road)
    .join(centres, "city_key", "left")
    .withColumn("distance_km", distance_km(col("centre_lat"), col("centre_lon"), col("lat"), col("lon")))
    .withColumn("location_ok", F.coalesce(col("distance_km") <= col("radius_km") * 1.42, F.lit(False)))
    .drop("centre_lat", "centre_lon", "radius_km")
)
display(happenings.groupBy("source", "category").count().orderBy("source", F.desc("count")))

In [0]:
# Cell 3: MERGE into silver.city_happenings and check
happenings.limit(0).write.format("delta").mode("ignore").saveAsTable(TARGET)   # create if missing, else do nothing

happenings.createOrReplaceTempView("happenings_updates")

spark.sql(f"""
    MERGE INTO {TARGET} AS t
    USING happenings_updates AS s
      ON t.happening_id = s.happening_id
    WHEN MATCHED AND s.version_ts >= t.version_ts THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"""
    SELECT source,
           count(*)                                           AS happenings,
           round(avg(distance_km), 1)                         AS avg_km,
           max(distance_km)                                   AS max_km,
           sum(CASE WHEN location_ok THEN 0 ELSE 0 + 1 END)   AS flagged
    FROM {TARGET}
    GROUP BY source
"""))

In [0]:
display(spark.sql("""
    SELECT source,
           sum(CASE WHEN location_ok = true  THEN 1 ELSE 0 END) AS ok,
           sum(CASE WHEN location_ok = false THEN 1 ELSE 0 END) AS not_ok,
           sum(CASE WHEN location_ok IS NULL THEN 1 ELSE 0 END) AS is_null,
           sum(CASE WHEN lat IS NULL         THEN 1 ELSE 0 END) AS no_coords
    FROM city_lens.silver.city_happenings
    GROUP BY source
"""))